In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler,LabelEncoder
import  pickle

In [3]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
## drop irrelevant columns 
data= data.drop(['RowNumber','CustomerId','Surname'],axis=1)

In [5]:
## lable encoding for categorical data
coded= LabelEncoder()
data['Gender']=coded.fit_transform(data['Gender'])

In [6]:
data['Gender'].head()

0    0
1    0
2    0
3    0
4    0
Name: Gender, dtype: int64

In [7]:
from sklearn.preprocessing import OneHotEncoder
One_hot= OneHotEncoder()
geography_encoded= One_hot.fit_transform(data[['Geography']]).toarray()
geography_encoded

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [8]:
geo_df= pd.DataFrame(geography_encoded,columns=One_hot.get_feature_names_out(['Geography']))
geo_df.head()

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [9]:
data=pd.concat([data.drop(columns=['Geography']),geo_df],axis=1)

data.head()


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [10]:
with open('preprocessor_gender.pkl','wb') as file:
    pickle.dump(coded,file)

with open('preprocessor_geography.pkl','wb') as file:
    pickle.dump(One_hot,file)

In [11]:
# dividing data into features and target variable\
X= data.drop(columns=['Exited'])
y= data['Exited']
X_train,X_test,y_train,y_test= train_test_split(X,y,test_size=0.2,random_state=42)  


In [12]:
Scaller = StandardScaler()
X_train_scaled= Scaller.fit_transform(X_train)
X_test_scaled= Scaller.transform(X_test)


In [13]:
with open('preprocessor.pkl','wb') as f:
    pickle.dump(One_hot,f)
    

# ANN _> Start's here 

In [14]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [15]:
model= Sequential([
    Dense(64,activation='relu',input_shape=(X_train_scaled.shape[1],)), # we are pssinh in tuple because it expects a tuple
    Dense(32,activation='relu'),                                          # shape[1] is the number of features/columns
    Dense(1,activation='sigmoid')   
])

2026-05-19 15:31:29.707004: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M2
2026-05-19 15:31:29.707197: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2026-05-19 15:31:29.707211: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2026-05-19 15:31:29.707680: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-19 15:31:29.708014: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [16]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


### param are combinations of wight , biares and inputs 

In [17]:
import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01) ## Adam ideology is to use different learning rates for different parameters and it also adapts the learning rate during training
loss=tensorflow.keras.losses.BinaryCrossentropy() ## loss function for binary classification problems
loss

# why we are writing keras here ? 
##   we write keras because BinaryCrossentropy belongs to the Keras module inside TensorFlow.
##   Keras is the high-level deep learning API inside TensorFlow.

In [18]:
compile= model.compile(optimizer=opt,loss=loss,metrics=['accuracy'])

# Whe we are compiling the model ? 
* we havn't connected modele with these objects that we just difined above 
* its mendetry step 

In [19]:
## Set up the Tensorboard
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard 

#callbacks_______> in short are functions that are called during the training process at certain points,
#such as at the end of an epoch or after a certain number of batches. They allow you to perform 
#specific actions or monitor the training process in real-time

#EarlyStopping_______> callback is used to stop the training process if the model's performance on a validation 
#set does not improve for a specified number of epochs. means earlystoping is callback funtion 


log_dir="logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

#log_dir is the directory where the TensorBoard logs will be saved. The logs will be organized 
# in subdirectories based on the current date and time, allowing you to easily track and compare 
# different training runs.how logs/fit/ ? works is that it creates a directory named "logs" and within 
# that, it creates a subdirectory named "fit".

tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)
#histogram_freq=1 means that histograms of the model's weights will be computed and saved to the logs
#every epoch.

# Why tensorboard ?
* to visulise ,monitor trianing 

In [28]:
EarlyStopping_callback= EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)
#monitor='val_loss' means that the callback will monitor the validation loss during training.
#patience=10 means that if the validation loss does not improve for 10 consecutive epochs, 
# the training will be stopped.
#restore_best_weights=True means that after stopping, the model's weights will be restored 
# to the values from the epoch with the best validation loss, rather than the last

In [30]:
history=model.fit(
    X_train_scaled,y_train,validation_data=(X_test_scaled,y_test)
    # validation_data is used to evaluate the model's performance on a separate dataset during training.
    ,epochs=100,callbacks=[tensorflow_callback,EarlyStopping_callback])

Epoch 1/100
250/250 [==============================] - 1s 4ms/step - loss: 23.5625 - accuracy: 0.7250 - val_loss: 361493.1562 - val_accuracy: 0.2015
Epoch 2/100
250/250 [==============================] - 1s 4ms/step - loss: 28.1422 - accuracy: 0.7170 - val_loss: 2141125.7500 - val_accuracy: 0.2115
Epoch 3/100
250/250 [==============================] - 1s 4ms/step - loss: 34.6598 - accuracy: 0.7256 - val_loss: 292571.4062 - val_accuracy: 0.1970
Epoch 4/100
250/250 [==============================] - 1s 4ms/step - loss: 40.1558 - accuracy: 0.7384 - val_loss: 1387441.6250 - val_accuracy: 0.1970
Epoch 5/100
250/250 [==============================] - 1s 4ms/step - loss: 42.7741 - accuracy: 0.7349 - val_loss: 1712438.2500 - val_accuracy: 0.1965
Epoch 6/100
250/250 [==============================] - 1s 4ms/step - loss: 55.3797 - accuracy: 0.7210 - val_loss: 5410381.5000 - val_accuracy: 0.1965
Epoch 7/100
250/250 [==============================] - 1s 4ms/step - loss: 55.7811 - accuracy: 0.7371 

In [32]:
model.save('churn_model.h5') # h5 is compatible with keras and tensorflow and it is a common
#format for saving deep learning models. It allows you to save the architecture, weights,
#  and training configuration of the model in a single file.

/Users/harshvardhansingh/Downloads/myannclasificationattempt/annclassy/lib/python3.11/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [37]:
# load tensorboard extension in jupyter notebook
#lonch tensorboard in jupyter notebook
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [38]:
%tensorboard --logdir logs/fit/20260519-153130 ## tensorboard is magic function that allows you to
#visualize the training process, including metrics like loss and accuracy, in real-time. 
# By pointing it to the log directory where the training logs are saved, you can monitor how 
# your model is performing during training.  /Users/harshvardhansingh/Downloads/myannclasificationattempt/logs/fit/20260519-153130

ERROR: Failed to launch TensorBoard (exited with 1).
Contents of stderr:
Traceback (most recent call last):
  File "/Users/harshvardhansingh/Downloads/myannclasificationattempt/annclassy/bin/tensorboard", line 3, in <module>
    from tensorboard.main import run_main
  File "/Users/harshvardhansingh/Downloads/myannclasificationattempt/annclassy/lib/python3.11/site-packages/tensorboard/main.py", line 27, in <module>
    from tensorboard import default
  File "/Users/harshvardhansingh/Downloads/myannclasificationattempt/annclassy/lib/python3.11/site-packages/tensorboard/default.py", line 30, in <module>
    import pkg_resources
ModuleNotFoundError: No module named 'pkg_resources'